# BM25 (Best Match 25)

BM25 is a **ranking function** used in information retrieval to score how relevant a document is to a search query. It is the algorithm behind many search engines (including Elasticsearch's default).


-> the 25 in name doesn't mean the cout of top-k. we can set it using n while calling hte funciton.

## Key Concepts

- **TF-IDF based** — rewards terms that appear frequently in a document but rarely in the corpus
- **Term Frequency Saturation** — caps the benefit of repeating a term (unlike raw TF-IDF)
- **Document Length Normalization** — penalizes longer documents so they don't dominate just by size

## Formula

```
score(D, Q) = Σ IDF(qᵢ) * [tf(qᵢ,D) * (k1 + 1)] / [tf(qᵢ,D) + k1 * (1 - b + b * |D|/avgdl)]
```

| Parameter | Typical Value | Role |
|-----------|--------------|------|
| `k1` | 1.2 – 2.0 | Term frequency saturation |
| `b` | 0.75 | Document length normalization |
| `avgdl` | computed | Average document length in corpus |

## BM25 vs Dense Embeddings

| | BM25 | Dense Embeddings |
|---|---|---|
| Type | Sparse / keyword-based | Dense / semantic |
| Understands synonyms | No | Yes |
| Speed | Fast | Slower |
| Good for | Exact terms, rare words, IDs | Meaning, paraphrasing |

In RAG, BM25 is often combined with vector search (**Hybrid Retrieval**) using **RRF (Reciprocal Rank Fusion)** to get both keyword precision and semantic understanding.

## Install Dependencies

In [ ]:
%pip install rank_bm25 -q

## Example: BM25 Search over a Small Document Corpus

In [ ]:
from rank_bm25 import BM25Okapi

# --- Corpus of documents ---
corpus = [
    "The quick brown fox jumps over the lazy dog",
    "Machine learning models require large amounts of training data",
    "BM25 is a ranking function used in information retrieval",
    "Neural networks are the backbone of modern deep learning",
    "Information retrieval systems rank documents by relevance to a query",
    "The lazy dog slept all afternoon near the fox",
]

# Tokenize: BM25 works on lists of tokens (words)
tokenized_corpus = [doc.lower().split() for doc in corpus]

# Build the BM25 index
bm25 = BM25Okapi(tokenized_corpus)

print("Corpus size:", len(corpus))
print("Sample tokenized doc:", tokenized_corpus[0])

In [ ]:
# --- Query 1: keyword match ---
query = "information retrieval ranking"
tokenized_query = query.lower().split()

scores = bm25.get_scores(tokenized_query)

print(f"Query: '{query}'\n")
print(f"{'Score':>8}  Document")
print("-" * 70)
for score, doc in sorted(zip(scores, corpus), reverse=True):
    print(f"{score:>8.4f}  {doc}")

In [ ]:
# --- Query 2: top-N retrieval (as used in RAG) ---
query2 = "fox and dog"
tokenized_query2 = query2.lower().split()

top_n = bm25.get_top_n(tokenized_query2, corpus, n=3)

print(f"Query: '{query2}'\n")
print("Top 3 results:")
for i, doc in enumerate(top_n, 1):
    print(f"  {i}. {doc}")

## Limitation: BM25 Doesn't Understand Semantics

BM25 relies purely on keyword overlap. It will score 0 for documents that are semantically related but use different words.

This is why **hybrid search** (BM25 + vector embeddings) is popular in RAG pipelines.

In [ ]:
# Semantic miss: query uses synonym "canine" — BM25 won't match "dog"
query3 = "canine animal sleeping"
tokenized_query3 = query3.lower().split()

scores3 = bm25.get_scores(tokenized_query3)

print(f"Query: '{query3}'\n")
print(f"{'Score':>8}  Document")
print("-" * 70)
for score, doc in sorted(zip(scores3, corpus), reverse=True):
    print(f"{score:>8.4f}  {doc}")

print("\n=> All scores are 0! BM25 can't match synonyms — this is where dense embeddings help.")